In [1]:
# !pip install python-dotenv timm huggingface_hub
from dotenv import load_dotenv
import os
load_dotenv()
hf_token = os.getenv("HF_TOKEN")


In [2]:

# ─────────────────────────────────────────────────────────────────────────────
# Virchow2 Extractor Class
# Output shape:  TenCrop → (10, 2560)  |  FiveCrop → (5, 2560)
# 2560 = 1280 class token + 1280 mean patch token
# ─────────────────────────────────────────────────────────────────────────────
import torch
import timm
from PIL import Image
from torchvision import transforms
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from timm.layers import SwiGLUPacked
from huggingface_hub import login


class Virchow2ExtractorCrops:
    """
    Virchow2 tile-level feature extractor with TenCrop / FiveCrop augmentation.

    Architecture : ViT-H/14  (632M params)
    Embedding    : 2560-d  (1280 class token + 1280 mean patch token)
    """

    def __init__(self, device: str = 'cuda', crop_size: int = 256):
        login(token=hf_token)

        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        print(f"[Virchow2] Loading model on {self.device} ...")

        # Load model exactly as the model card specifies
        self.model = timm.create_model(
            "hf-hub:paige-ai/Virchow2",
            pretrained=True,
            mlp_layer=SwiGLUPacked,
            act_layer=torch.nn.SiLU,
        ).to(self.device)
        self.model.eval()
        print("[Virchow2] Model loaded ✓")

        # Resolve normalization stats directly from model pretrained_cfg
        # (ImageNet stats: mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
        data_cfg = resolve_data_config(self.model.pretrained_cfg, model=self.model)
        mean = data_cfg["mean"]
        std  = data_cfg["std"]

        # Per-crop transform: resize each crop → normalize
        self.eval_transform = transforms.Compose([
            transforms.Resize((224, 224)),  # ViT-H/14 expects 224×224 crops
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ])

        self.fivecrop_transform = transforms.Compose([
            transforms.FiveCrop(crop_size),
            transforms.Lambda(lambda crops: torch.stack(
                [self.eval_transform(c) for c in crops]
            )),
        ])

        self.tencrop_transform = transforms.Compose([
            transforms.TenCrop(crop_size),
            transforms.Lambda(lambda crops: torch.stack(
                [self.eval_transform(c) for c in crops]
            )),
        ])

    def _load_image(self, image) -> Image.Image:
        if isinstance(image, str):
            return Image.open(image).convert("RGB")
        return image.convert("RGB")

    @torch.inference_mode()
    def _forward(self, crops_tensor: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for a batch of crops.
        crops_tensor : (N, 3, 224, 224)
        returns      : (N, 2560)  fp32
        """
        crops_tensor = crops_tensor.to(self.device)
        with torch.autocast(device_type=self.device.type, dtype=torch.float16):
            output = self.model(crops_tensor)   # (N, 261, 1280)

        # Tokens 1–4 are register tokens → skip them
        class_token  = output[:, 0]      # (N, 1280)
        patch_tokens = output[:, 5:]     # (N, 256, 1280)

        # Concatenate class token + mean patch token → 2560-d embedding
        embedding = torch.cat([class_token, patch_tokens.mean(dim=1)], dim=-1)  # (N, 2560)
        return embedding.cpu().float()

    def extract_fivecrop_features(self, image) -> torch.Tensor:
        """
        Extract features from 5 crops of an image.
        Returns tensor of shape (5, 2560).
        """
        img = self._load_image(image)
        crops_tensor = self.fivecrop_transform(img)   # (5, 3, 224, 224)
        return self._forward(crops_tensor)

    def extract_tencrop_features(self, image) -> torch.Tensor:
        """
        Extract features from 10 crops (5 original + 5 horizontal flips).
        Returns tensor of shape (10, 2560).
        """
        img = self._load_image(image)
        crops_tensor = self.tencrop_transform(img)    # (10, 3, 224, 224)
        return self._forward(crops_tensor)



In [3]:

# ─────────────────────────────────────────────────────────────────────────────
#  Quick sanity check (verify shapes before running full pipeline)
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np

extractor = Virchow2ExtractorCrops(device='cuda')

dummy_img = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))

feat_5  = extractor.extract_fivecrop_features(dummy_img)
feat_10 = extractor.extract_tencrop_features(dummy_img)

print(f"FiveCrop  shape : {feat_5.shape}   → expected (5,  2560)")
print(f"TenCrop   shape : {feat_10.shape}  → expected (10, 2560)")


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful
[Virchow2] Loading model on cuda ...
[Virchow2] Model loaded ✓
FiveCrop  shape : torch.Size([5, 2560])   → expected (5,  2560)
TenCrop   shape : torch.Size([10, 2560])  → expected (10, 2560)


c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\layers\attention.py:80: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  x = F.scaled_dot_product_attention(


In [3]:

# ─────────────────────────────────────────────────────────────────────────────
# Full pipeline 
# ─────────────────────────────────────────────────────────────────────────────
import glob
import pandas as pd
from tqdm import tqdm


def run_full_pipeline_Virchow2_crops(
    patch_data_dir: str,
    extractor: Virchow2ExtractorCrops,
    non_white_csv: str,
    output_root: str,
    mode: str = "fivecrop",      # "tencrop" → (10, 2560)  |  "fivecrop" → (5, 2560)
):
    assert mode in ("tencrop", "fivecrop"), "mode must be 'tencrop' or 'fivecrop'"
    extract_fn = (
        extractor.extract_tencrop_features if mode == "tencrop"
        else extractor.extract_fivecrop_features
    )

    # Load valid (non-white) patch names
    non_white_df = pd.read_csv(non_white_csv)
    valid_patch_names = set(non_white_df["patch_name"].astype(str))
    print(f"[Pipeline] Valid patches in CSV : {len(valid_patch_names):,}")
    print(f"[Pipeline] Mode                 : {mode}")
    print(f"[Pipeline] Output root          : {output_root}\n")

    slide_dirs = sorted(glob.glob(os.path.join(patch_data_dir, "*")))

    for slide_dir in tqdm(slide_dirs, desc="Slides", unit="slide", position=0):
        slide_name = os.path.basename(slide_dir)
        feat_dir   = os.path.join(output_root, slide_name)
        os.makedirs(feat_dir, exist_ok=True)

        # Filter to valid non-white patches only
        all_patches   = glob.glob(os.path.join(slide_dir, "*.png"))
        valid_patches = [p for p in all_patches
                         if os.path.basename(p) in valid_patch_names]

        if not valid_patches:
            tqdm.write(f"  ❌  No valid patches for {slide_name}")
            continue

        # Skip already-processed patches (resume-safe)
        done = {os.path.splitext(f.name)[0]
                for f in os.scandir(feat_dir) if f.name.endswith(".pt")}
        to_process = [p for p in valid_patches
                      if os.path.splitext(os.path.basename(p))[0] not in done]

        if not to_process:
            tqdm.write(f"  ⏭️  Skipped {slide_name} ({len(valid_patches):,} patches already done)")
            continue

        for patch_path in tqdm(to_process, desc=f"  {slide_name}",
                               unit="patch", position=1, leave=False):
            patch_name = os.path.splitext(os.path.basename(patch_path))[0]
            save_path  = os.path.join(feat_dir, patch_name + ".pt")
            try:
                features = extract_fn(patch_path)   # (10, 2560) or (5, 2560)
                torch.save(features, save_path)
            except Exception as e:
                tqdm.write(f"  ❌  Error on {patch_path}: {e}")

        tqdm.write(f"  ✅  {slide_name} — processed {len(to_process):,} new "
                   f"(total valid: {len(valid_patches):,})")



In [5]:
extractor = Virchow2ExtractorCrops(device='cuda')

run_full_pipeline_Virchow2_crops(
    patch_data_dir = r"D:\Aamir Gulzar\KSA_project2\dataset\patch_data",
    extractor      = extractor,
    non_white_csv  = r"D:\Aamir Gulzar\KSA_project2\dataset\final_merged_without_white.csv",
    output_root    = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Virchow2_fivecrop",
    mode           = "fivecrop",     # switch to "fivecrop" if needed
)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful
[Virchow2] Loading model on cuda ...
[Virchow2] Model loaded ✓
[Pipeline] Valid patches in CSV : 1,377,933
[Pipeline] Mode                 : fivecrop
[Pipeline] Output root          : D:\Aamir Gulzar\KSA_project2\dataset\Features\Virchow2_fivecrop



Slides:   0%|          | 1/417 [01:37<11:14:50, 97.33s/slide]

  ✅  TCGA-3L-AA1B_nonMSIH — processed 1,878 new (total valid: 1,878)


Slides:   0%|          | 2/417 [05:15<19:23:19, 168.19s/slide]

  ✅  TCGA-4N-A93T_nonMSIH — processed 4,240 new (total valid: 4,240)


...

  ✅  TCGA-SS-A7HO_nonMSIH — processed 5,746 new (total valid: 5,746)


Slides: 100%|█████████▉| 416/417 [19:03:57<04:07, 247.35s/slide]

  ✅  TCGA-T9-A92H_nonMSIH — processed 5,462 new (total valid: 5,462)


Slides: 100%|██████████| 417/417 [19:08:21<00:00, 165.23s/slide]


  ✅  TCGA-WS-AB45_MSIH — processed 4,532 new (total valid: 4,532)


In [4]:
extractor = Virchow2ExtractorCrops(device='cuda')

run_full_pipeline_Virchow2_crops(
    patch_data_dir = r"D:\Aamir Gulzar\KSA_project2\paip_data\patch_data",
    extractor      = extractor,
    non_white_csv  = r"D:\Aamir Gulzar\KSA_project2\paip_data\final_merged_without_white.csv",
    output_root    = r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\Virchow2_fivecrop_Updated",
    mode           = "fivecrop",     # switch to "fivecrop" if needed
)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful
[Virchow2] Loading model on cuda ...
[Virchow2] Model loaded ✓
[Pipeline] Valid patches in CSV : 271,322
[Pipeline] Mode                 : fivecrop
[Pipeline] Output root          : D:\Aamir Gulzar\KSA_project2\paip_data\Features\Virchow2_fivecrop_Updated



Slides:   3%|▎         | 2/73 [00:00<00:03, 19.32slide/s]

  ⏭️  Skipped training_data_01_MSIH (3,196 patches already done)
  ⏭️  Skipped training_data_02_nonMSIH (4,412 patches already done)
  ⏭️  Skipped training_data_03_nonMSIH (1,955 patches already done)


Slides:   3%|▎         | 2/73 [00:00<00:03, 19.32slide/s]

  ⏭️  Skipped training_data_04_nonMSIH (3,198 patches already done)


...

  ✅  validation_data_29_MSIH — processed 4,171 new (total valid: 4,171)


Slides:  99%|█████████▊| 72/73 [10:36<01:25, 85.60s/slide]

  ✅  validation_data_30_nonMSIH — processed 4,118 new (total valid: 4,118)


Slides: 100%|██████████| 73/73 [15:01<00:00, 12.34s/slide] 

  ✅  validation_data_31_nonMSIH — processed 4,810 new (total valid: 4,810)


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL — Environment Compatibility Check for Virchow2
# Run this in your existing environment BEFORE running Virchow2
# ─────────────────────────────────────────────────────────────────────────────
import importlib, subprocess, sys

def check(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"  ✓  {pkg:<20} {version}")
        return True
    except ImportError:
        print(f"  ✗  {pkg:<20} NOT FOUND")
        return False

print("=" * 45)
print("  Virchow2 Environment Check")
print("=" * 45)

ok_torch = check("torch")
ok_timm  = check("timm")
ok_hf    = check("huggingface_hub")
check("torchvision")
check("PIL", "PIL")

# Extra: check timm version is >= 0.9.11 (required by Virchow2 model card)
import timm
from packaging import version
timm_ok = version.parse(timm.__version__) >= version.parse("0.9.11")
print(f"\n  timm >= 0.9.11 required   → {'✓ OK' if timm_ok else '✗ UPGRADE NEEDED: pip install timm --upgrade'}")

# Check SwiGLUPacked is available (key Virchow2 dependency)
try:
    from timm.layers import SwiGLUPacked
    print(f"  SwiGLUPacked available    → ✓ OK")
except ImportError:
    print(f"  SwiGLUPacked available    → ✗ NOT FOUND — upgrade timm: pip install timm --upgrade")

# Check CUDA
import torch
print(f"\n  CUDA available            → {'✓  ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '✗  CPU only'}")
print(f"  PyTorch version           → {torch.__version__}")

print("\n" + "=" * 45)
if timm_ok:
    print("  ✅  Your environment is ready for Virchow2.")
    print("  No new environment needed.")
else:
    print("  ⚠️  Run:  pip install timm --upgrade")
    print("  Then re-run this cell to confirm.")
print("=" * 45)

  Virchow2 Environment Check
  ✓  torch                2.2.0
  ✓  timm                 1.0.17
  ✓  huggingface_hub      0.23.4
  ✓  torchvision          0.17.0
  ✓  PIL                  10.0.1

  timm >= 0.9.11 required   → ✓ OK
  SwiGLUPacked available    → ✓ OK

  CUDA available            → ✓  NVIDIA GeForce RTX 4090
  PyTorch version           → 2.2.0

  ✅  Your environment is ready for Virchow2.
  No new environment needed.


In [5]:
import torch
import random
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────────────
FEATURE_DIR = r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\Virchow2_fivecrop_Updated"
N_SAMPLES   = 10   # how many random files to peek at
# ──────────────────────────────────────────────────────────────────────────────

all_files = list(Path(FEATURE_DIR).rglob("*.pt"))
sampled   = random.sample(all_files, min(N_SAMPLES, len(all_files)))

print(f"{'File':<60} {'Shape':<15} {'Crops'}")
print("─" * 90)
for path in sampled:
    tensor = torch.load(path, map_location="cpu", weights_only=True)
    crops  = "FiveCrop" if tensor.shape[0] == 5 else "TenCrop" if tensor.shape[0] == 10 else "Unknown"
    print(f"{path.name:<60} {str(tuple(tensor.shape)):<15} {crops}")
# ```

# **Output looks like:**
# ```
# File                                                         Shape           Crops
# ──────────────────────────────────────────────────────────────────────────────────────────
# patch_00123.pt                                               (5, 2560)       FiveCrop
# patch_00456.pt                                               (5, 2560)       FiveCrop
# ...
# ```

# ---

# ## Without Code

# Yes! Just open **Windows Explorer**, navigate to your feature folder, and open any `.pt` file is **not** human readable. But you can check **file size** instead:
# ```
# FiveCrop  (5, 2560)  float32 → 5  × 2560 × 4 bytes ≈  51 KB
# TenCrop  (10, 2560)  float32 → 10 × 2560 × 4 bytes ≈ 102 KB

File                                                         Shape           Crops
──────────────────────────────────────────────────────────────────────────────────────────
training_data_07_nonMSIH_x43088_y20721_patch02493.pt         (5, 2560)       FiveCrop
training_data_18_nonMSIH_x21248_y0_patch02204.pt             (5, 2560)       FiveCrop
training_data_14_nonMSIH_x31232_y33028_patch02103.pt         (5, 2560)       FiveCrop
training_data_14_nonMSIH_x35840_y30980_patch02332.pt         (5, 2560)       FiveCrop
training_data_31_nonMSIH_x42048_y11776_patch05376.pt         (5, 2560)       FiveCrop
validation_data_18_nonMSIH_x18305_y17408_patch00554.pt       (5, 2560)       FiveCrop
validation_data_12_nonMSIH_x10064_y28784_patch00915.pt       (5, 2560)       FiveCrop
training_data_40_nonMSIH_x40528_y36242_patch01454.pt         (5, 2560)       FiveCrop
validation_data_01_nonMSIH_x39504_y13824_patch04940.pt       (5, 2560)       FiveCrop
training_data_43_nonMSIH_x42736_y13810_patch01750.pt

In [8]:
import torch
import random
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from collections import Counter
from IPython.display import display

# ─── CONFIG ───────────────────────────────────────────────────────────────────
AGG_DIR        = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation\Virchow2"
EXPECTED_SHAPE = torch.Size([14, 2560])
FEATURE_DIM    = 2560
SAMPLE_RATE    = 1.0        # 1.0 = check ALL files
all_groups     = ['ADI', 'DEB', 'LYM', 'PLC', 'LYA', 'MUC', 'MUS', 'NORM',
                  'ADE', 'STR', 'MES', 'CAR', 'PDC', 'SIG']
# ──────────────────────────────────────────────────────────────────────────────

def validate_aggregated_file(path: Path) -> dict:
    result = {
        "file"         : path.name,
        "slide_name"   : path.stem,
        "ok"           : False,
        "error"        : None,
        "shape"        : None,
        "dtype"        : None,
        "file_size_kb" : round(path.stat().st_size / 1024, 2),
        "has_nan"      : False,
        "has_inf"      : False,
        "n_zero_rows"  : 0,
        "zero_groups"  : [],
        "min"          : None,
        "max"          : None,
        "mean"         : None,
        "std"          : None,
    }

    try:
        tensor = torch.load(path, map_location="cpu", weights_only=True)

        # ── 1. Must be a tensor ───────────────────────────────────────────────
        if not isinstance(tensor, torch.Tensor):
            result["error"] = f"Not a tensor: {type(tensor)}"
            return result

        result["shape"] = tuple(tensor.shape)
        result["dtype"] = str(tensor.dtype)

        # ── 2. Shape must be (14, 2560) ───────────────────────────────────────
        if tensor.shape != EXPECTED_SHAPE:
            result["error"] = f"Wrong shape {tuple(tensor.shape)}, expected {tuple(EXPECTED_SHAPE)}"
            return result

        t = tensor.float()

        # ── 3. NaN / Inf ──────────────────────────────────────────────────────
        result["has_nan"] = bool(torch.isnan(t).any())
        result["has_inf"] = bool(torch.isinf(t).any())

        if result["has_nan"]:
            result["error"] = "Contains NaN"
            return result
        if result["has_inf"]:
            result["error"] = "Contains Inf"
            return result

        # ── 4. Per-row (per-group) zero check ─────────────────────────────────
        zero_mask             = (t == 0).all(dim=1)
        zero_indices          = zero_mask.nonzero(as_tuple=True)[0].tolist()
        result["n_zero_rows"] = len(zero_indices)
        result["zero_groups"] = [all_groups[i] for i in zero_indices]

        # ── 5. Value stats ────────────────────────────────────────────────────
        result["min"]  = round(float(t.min()), 6)
        result["max"]  = round(float(t.max()), 6)
        result["mean"] = round(float(t.mean()), 6)
        result["std"]  = round(float(t.std()), 6)

        result["ok"] = True

    except Exception as e:
        result["error"] = str(e)

    return result


def run_aggregation_check(agg_dir: str, sample_rate: float = SAMPLE_RATE):

    root      = Path(agg_dir)
    all_files = list(root.glob("*.pt"))
    total     = len(all_files)

    print(f"📂 Directory      : {root}")
    print(f"   .pt files found : {total:,}\n")

    if total == 0:
        print("❌ No .pt files found. Check your AGG_DIR path.")
        return None

    if sample_rate < 1.0:
        n_sample = max(1, int(total * sample_rate))
        sampled  = random.sample(all_files, n_sample)
        print(f"Sampling {sample_rate*100:.1f}% → checking {n_sample:,} files\n")
    else:
        sampled = all_files
        print(f"Checking ALL {total:,} files\n")

    # ── Sequential validation ─────────────────────────────────────────────────
    results = []
    for path in tqdm(sampled, desc="Validating"):
        results.append(validate_aggregated_file(path))

    df      = pd.DataFrame(results)
    ok      = df[df["ok"] == True]
    corrupt = df[df["ok"] == False]

    # ─────────────────────────────────────────────────────────────────────────
    # REPORT
    # ─────────────────────────────────────────────────────────────────────────
    print(f"\n{'═'*65}")
    print(f"  AGGREGATION VALIDATION REPORT")
    print(f"{'═'*65}")
    print(f"  Total files checked : {len(sampled):,}")
    print(f"  ✓ Clean             : {len(ok):,}")
    print(f"  ✗ Corrupt / Bad     : {len(corrupt):,}")
    print(f"{'═'*65}\n")

    # ── Shape distribution ────────────────────────────────────────────────────
    print(f"{'─'*65}")
    print(f"  SHAPE DISTRIBUTION  (expected: {tuple(EXPECTED_SHAPE)})")
    print(f"{'─'*65}")
    shape_counts = df["shape"].value_counts().reset_index()
    shape_counts.columns = ["Shape", "Count"]
    shape_counts["Match"] = shape_counts["Shape"].apply(
        lambda s: "✓" if str(s) == str(tuple(EXPECTED_SHAPE)) else "✗"
    )
    display(shape_counts)

    # ── Dtype distribution ────────────────────────────────────────────────────
    print(f"\n{'─'*65}")
    print(f"  DTYPE DISTRIBUTION")
    print(f"{'─'*65}")
    display(df["dtype"].value_counts().reset_index().rename(
        columns={"dtype": "DType", "count": "Count"}
    ))

    # ── File size stats ───────────────────────────────────────────────────────
    print(f"\n{'─'*65}")
    print(f"  FILE SIZE STATS (KB)  — expected ≈ {round(14 * 2560 * 4 / 1024, 1)} KB per file")
    print(f"{'─'*65}")
    display(df["file_size_kb"].describe().round(2).to_frame().T)

    # ── Zero group analysis ───────────────────────────────────────────────────
    if len(ok) > 0:
        print(f"\n{'─'*65}")
        print(f"  ZERO GROUP ANALYSIS  (groups with no patches = zero vector)")
        print(f"{'─'*65}")
        slides_with_zeros = (ok["n_zero_rows"] > 0).sum()
        print(f"  Slides with ≥1 zero group : {slides_with_zeros:,} / {len(ok):,}")
        print(f"  Avg zero groups per slide : {ok['n_zero_rows'].mean():.2f}\n")

        group_zero_counts = Counter()
        for zero_list in ok["zero_groups"]:
            group_zero_counts.update(zero_list)

        df_zero = pd.DataFrame(
            group_zero_counts.most_common(),
            columns=["Group", "Slides Where Zero"]
        )
        df_zero["% of slides"] = (df_zero["Slides Where Zero"] / len(ok) * 100).round(2)
        display(df_zero)

    # ── Value stats ───────────────────────────────────────────────────────────
    if len(ok) > 0:
        print(f"\n{'─'*65}")
        print(f"  TENSOR VALUE STATS  (across {len(ok):,} clean files)")
        print(f"{'─'*65}")
        display(ok[["min", "max", "mean", "std"]].describe().round(6))

    # ── Corrupt file details ──────────────────────────────────────────────────
    if len(corrupt) > 0:
        print(f"\n{'─'*65}")
        print(f"  ✗ CORRUPT FILES")
        print(f"{'─'*65}")
        error_summary = corrupt["error"].value_counts().reset_index()
        error_summary.columns = ["Error Type", "Count"]
        display(error_summary)
        print("\n  Corrupt file details:")
        display(corrupt[["file", "shape", "dtype", "error", "file_size_kb"]])

        out_csv = root / "corrupt_aggregated.csv"
        corrupt.to_csv(out_csv, index=False)
        print(f"\n  💾 Corrupt list saved → {out_csv}")
    else:
        print("\n✅ All aggregated files are valid and correct!")

    return df


# ─── RUN ──────────────────────────────────────────────────────────────────────
df_results = run_aggregation_check(
    agg_dir     = AGG_DIR,
    sample_rate = SAMPLE_RATE,
)

📂 Directory      : D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation\Virchow2
   .pt files found : 73

Checking ALL 73 files



Validating: 100%|██████████| 73/73 [00:00<00:00, 124.03it/s]


═════════════════════════════════════════════════════════════════
  AGGREGATION VALIDATION REPORT
═════════════════════════════════════════════════════════════════
  Total files checked : 73
  ✓ Clean             : 73
  ✗ Corrupt / Bad     : 0
═════════════════════════════════════════════════════════════════

─────────────────────────────────────────────────────────────────
  SHAPE DISTRIBUTION  (expected: (14, 2560))
─────────────────────────────────────────────────────────────────


,Shape,Count,Match
0,"(14, 2560)",73,✓



─────────────────────────────────────────────────────────────────
  DTYPE DISTRIBUTION
─────────────────────────────────────────────────────────────────


...


─────────────────────────────────────────────────────────────────
  ZERO GROUP ANALYSIS  (groups with no patches = zero vector)
─────────────────────────────────────────────────────────────────
  Slides with ≥1 zero group : 9 / 73
  Avg zero groups per slide : 0.27



,Group,Slides Where Zero,% of slides
0,CAR,5,6.85
1,ADE,3,4.11
2,ADI,2,2.74
3,PLC,2,2.74
4,DEB,1,1.37
5,LYA,1,1.37
6,MUC,1,1.37
7,NORM,1,1.37
8,STR,1,1.37
9,MES,1,1.37



─────────────────────────────────────────────────────────────────
  TENSOR VALUE STATS  (across 73 clean files)
─────────────────────────────────────────────────────────────────


,min,max,mean,std
count,73.000000,73.000000,73.000000,73.000000
mean,-26.001900,7.976549,-0.007163,0.650084
std,4.715391,2.615072,0.001488,0.048217
min,-34.646152,5.460355,-0.010651,0.351733
25%,-29.499220,6.186872,-0.008037,0.629106
50%,-26.655603,6.834763,-0.007125,0.657640
75%,-23.902889,9.147350,-0.006480,0.672811
max,-13.142479,20.258640,0.000433,0.749826



✅ All aggregated files are valid and correct!
